# LAB 4: Vietnamese Spell-checking Pipeline using PhoBERT

This notebook implements a complete Vietnamese spell-checking system using PhoBERT (Vietnamese BERT) for intelligent error detection and correction.

## 0. Installation and Imports

First, let's install all required dependencies and import necessary libraries.

In [ ]:
# Install required packages (only needed the first time)
%pip install -q transformers torch


In [ ]:
# Import necessary libraries
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
import numpy as np
import os
import re
import difflib
import unicodedata
from typing import List, Tuple

## 1. Model Loading

Load PhoBERT model with comprehensive fallback strategy to ensure compatibility.

In [ ]:
os.environ.pop('HUGGINGFACE_HUB_TOKEN', None)
os.environ.pop('HF_TOKEN', None)

# Model name can be overridden by env var (e.g. a local folder) - default: PhoBERT base
MODEL_NAME = os.environ.get("PHOBERT_MODEL", "vinai/phobert-base")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
model.eval()
print(f"{MODEL_NAME} loaded successfully! ({type(model).__name__})")


## 2. Masking Words

Function to create masked versions of sentences for spell-checking analysis.

In [ ]:
def mask_words_sequentially(sentence: str) -> List[Tuple[str, str, int]]:
    """
    Masks each word in a sentence sequentially for spell-checking.
    
    Args:
        sentence (str): Input sentence to process
        
    Returns:
        List[Tuple[str, str, int]]: List of (masked_sentence, original_word, word_position)
    """
    # Tokenize the sentence (simple word-based tokenization)
    words = sentence.split()
    results = []
    
    for i, word in enumerate(words):
        # TODO: Implement masking logic
        masked_words = words.copy()
        masked_words[i] = "[MASK]"
        masked_sentence = " ".join(masked_words)
        results.append((masked_sentence, word, i))
        # END OF TODO
    return results

🧪 Testing mask_words_sequentially function:
Position 1: [MASK] mì là một loại đồ án của Việt Nam
Original word: 'Bánh'
--------------------------------------------------
Position 2: Bánh [MASK] là một loại đồ án của Việt Nam
Original word: 'mì'
--------------------------------------------------
Position 3: Bánh mì [MASK] một loại đồ án của Việt Nam
Original word: 'là'
--------------------------------------------------


In [ ]:
# Test the function
print("🧪 Testing mask_words_sequentially function:")
test_sentence = "Bánh mì là một loại đồ án của Việt Nam"
masked_results = mask_words_sequentially(test_sentence)

for masked_sentence, original_word, position in masked_results[:3]:  # Show first 3 examples
    print(f"Position {position+1}: {masked_sentence}")
    print(f"Original word: '{original_word}'")
    print("-" * 50)

## 3. Calculate Word Probability

Function to calculate how likely a word fits in a masked position using PhoBERT.

In [ ]:
def calculate_word_probability(masked_sentence: str, original_word: str) -> float:
    """
    Calculate the probability of a word fitting in a masked position using BERT.
    
    Args:
        masked_sentence (str): Sentence with [MASK] token
        original_word (str): The original word to check probability for
        
    Returns:
        float: Probability score between 0 and 1
    """
    try:
        # TODO: Implement probability calculation logic for the original_word in the masked_sentence
        masked_sentence = masked_sentence.replace("[MASK]", tokenizer.mask_token)
        inputs = tokenizer(masked_sentence, return_tensors="pt", truncation=True, max_length=256)
        mask_positions = (inputs["input_ids"][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]
        if len(mask_positions) == 0:
            return 0.0
        with torch.no_grad():
            logits = model(**inputs).logits
        probs = torch.softmax(logits[0, mask_positions[0]], dim=-1)
        # A word may be split into several sub-word tokens -> average their probabilities
        word_ids = tokenizer.encode(original_word, add_special_tokens=False)
        if not word_ids:
            return 0.0
        probability = sum(probs[t].item() for t in word_ids) / len(word_ids)
        # END OF TODO
        return probability
    
    except Exception as e:
        print(f"Error calculating probability for '{original_word}': {e}")
        return 0.0

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


🧪 Testing calculate_word_probability function:
Probability for correct word 'ăn': 0.432661
Probability for incorrect word 'án': 0.000001
Difference: 0.432660


In [ ]:
# Test the function
print("🧪 Testing calculate_word_probability function:")
test_masked_sentence = "Bánh mì là một loại đồ [MASK] của Việt Nam"
prob_correct = calculate_word_probability(test_masked_sentence, "ăn")
prob_incorrect = calculate_word_probability(test_masked_sentence, "án")

print(f"Probability for correct word 'ăn': {prob_correct:.6f}")
print(f"Probability for incorrect word 'án': {prob_incorrect:.6f}")
print(f"Difference: {prob_correct - prob_incorrect:.6f}")

## 4. Main Function: Spell Checking

Function to analyze entire sentences and identify potential spelling errors.

In [ ]:
def check_spelling_errors(sentence: str, threshold: float = 0.00001) -> List[Tuple[str, float, bool]]:
    """
    Check for spelling errors in a sentence using probability threshold.
    
    Args:
        sentence (str): Input sentence to check
        threshold (float): Probability threshold below which a word is considered misspelled
        
    Returns:
        List[Tuple[str, float, bool]]: List of (word, probability, is_error)
    """
    words = sentence.split()
    results = []
    
    print(f"Analyzing sentence: '{sentence}'")
    print(f"Using threshold: {threshold}")
    print("=" * 60)
    
    for i, word in enumerate(words):
        # TODO: Create masked sentence
        masked_sentence = " ".join(words[:i] + ["[MASK]"] + words[i+1:])
        # TODO: Calculate probability (prob)
        prob = calculate_word_probability(masked_sentence, word)
        # END OF TODO
        is_error = prob < threshold
        
        results.append((word, prob, is_error))
        status = "❌ POTENTIAL ERROR" if is_error else "✅ OK"
        confidence = "HIGH" if prob > 0.01 else "MEDIUM" if prob > 0.001 else "LOW"
        print(f"Word: '{word:12}' | Prob: {prob:.6f} | Confidence: {confidence:6} | {status}")
    
    print("=" * 60)
    errors_found = sum(1 for _, _, is_error in results if is_error)
    total_words = len(words)
    print(f"📊 Summary: {errors_found}/{total_words} words flagged as potential errors ({errors_found/total_words*100:.1f}%)")
    return results

🧪 Testing check_spelling_errors function:

📝 Test 1 - Correct sentence:
🔍 Analyzing sentence: 'Bánh mì là một loại đồ ăn của Việt Nam'
📏 Using threshold: 0.001
🤖 Model: RobertaForMaskedLM
Word: 'Bánh        ' | Prob: 0.072656 | Confidence: HIGH   | ✅ OK
Word: 'mì          ' | Prob: 0.000433 | Confidence: LOW    | ❌ POTENTIAL ERROR
Word: 'là          ' | Prob: 0.989855 | Confidence: HIGH   | ✅ OK
Word: 'một         ' | Prob: 0.995947 | Confidence: HIGH   | ✅ OK
Word: 'Bánh        ' | Prob: 0.072656 | Confidence: HIGH   | ✅ OK
Word: 'mì          ' | Prob: 0.000433 | Confidence: LOW    | ❌ POTENTIAL ERROR
Word: 'là          ' | Prob: 0.989855 | Confidence: HIGH   | ✅ OK
Word: 'một         ' | Prob: 0.995947 | Confidence: HIGH   | ✅ OK
Word: 'loại        ' | Prob: 0.526087 | Confidence: HIGH   | ✅ OK
Word: 'đồ          ' | Prob: 0.004811 | Confidence: MEDIUM | ✅ OK
Word: 'loại        ' | Prob: 0.526087 | Confidence: HIGH   | ✅ OK
Word: 'đồ          ' | Prob: 0.004811 | Confidence: MEDIUM |

In [ ]:
# Test the function
print("🧪 Testing check_spelling_errors function:")
print()

# Test with correct sentence
correct_sentence = "Bánh mì là một loại đồ ăn của Việt Nam"
print("📝 Test 1 - Correct sentence:")
results1 = check_spelling_errors(correct_sentence, threshold=0.001)
# Test with a sentence containing an error ("án" should be "ăn")
print()
print("📝 Test 2 - Sentence with error:")
results2 = check_spelling_errors("Bánh mì là một loại đồ án của Việt Nam", threshold=0.001)


## 5. Prediction Function: Top Suggestions

Function to get top word predictions for masked positions.

In [ ]:
def get_top_predictions(masked_sentence: str, top_k: int = 5) -> List[Tuple[str, float]]:
    """
    Get top predictions for a masked position.
    
    Args:
        masked_sentence (str): Sentence with [MASK] token
        top_k (int): Number of top predictions to return
        
    Returns:
        List[Tuple[str, float]]: List of (predicted_word, probability)
    """
    try:
        # TODO: Replace [MASK] with the actual mask token
        masked_sentence = masked_sentence.replace("[MASK]", tokenizer.mask_token)
        # TODO: Tokenize and get predictions
        inputs = tokenizer(masked_sentence, return_tensors="pt", truncation=True, max_length=256)
        mask_positions = (inputs["input_ids"][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]
        if len(mask_positions) == 0:
            return []
        with torch.no_grad():
            logits = model(**inputs).logits
        probs = torch.softmax(logits[0, mask_positions[0]], dim=-1)
        # TODO: Get top k predictions
        top_probs, top_ids = torch.topk(probs, top_k)
        results = []
        for p, t in zip(top_probs.tolist(), top_ids.tolist()):
            word = tokenizer.convert_ids_to_tokens(t).replace("@@", "")
            results.append((word, p))

        # END OF TODO
        return results
    
    except Exception as e:
        print(f"❌ Error getting predictions: {e}")
        return []

🧪 Testing get_top_predictions function:
For sentence: 'Bánh mì là một loại đồ [MASK] của Việt Nam'
Top predictions:
  1. 'ăn' (probability: 0.432661)
  2. 'tráng_miệng' (probability: 0.123275)
  3. 'uống' (probability: 0.087558)
  4. 'ngọt' (probability: 0.053298)
  5. 'ẩm_thực' (probability: 0.027972)


In [ ]:
# Test the function
print("🧪 Testing get_top_predictions function:")
test_masked = "Bánh mì là một loại đồ [MASK] của Việt Nam"
predictions = get_top_predictions(test_masked, top_k=5)

print(f"For sentence: '{test_masked}'")
print("Top predictions:")
for i, (word, prob) in enumerate(predictions, 1):
    print(f"  {i}. '{word}' (probability: {prob:.6f})")

## 6. Find similar words

Helper function for similarity comparison by removing Vietnamese diacritical marks.

### 6.1. Utl: Remove accents

In [8]:
def remove_accents(text: str) -> str:
    """
    Remove Vietnamese accents for similarity comparison.
    
    Args:
        text (str): Input text with Vietnamese accents
        
    Returns:
        str: Text without accents
    """
    return ''.join(c for c in unicodedata.normalize('NFD', text) 
                   if unicodedata.category(c) != 'Mn')

# Test the function
print("🧪 Testing remove_accents function:")
test_words = ["ăn", "án", "Việt", "việt", "phở", "pho"]

for word in test_words:
    no_accent = remove_accents(word)
    print(f"'{word}' -> '{no_accent}'")

🧪 Testing remove_accents function:
'ăn' -> 'an'
'án' -> 'an'
'Việt' -> 'Viet'
'việt' -> 'viet'
'phở' -> 'pho'
'pho' -> 'pho'


## 6.2. Find Similar Words

In [ ]:
def find_similar_words(target_word: str, predictions: List[str], threshold: float = 0.6) -> List[Tuple[str, float]]:
    """
    Find similar words based on string similarity.
    
    Args:
        target_word (str): The word to find similarities for
        predictions (List[str]): List of candidate words
        threshold (float): Minimum similarity threshold
        
    Returns:
        List[Tuple[str, float]]: List of (word, similarity_score) pairs
    """
    similar_words = []
    target_clean = remove_accents(target_word.lower())
    
    for pred in predictions:
        pred_clean = remove_accents(pred.lower())
        similarity = difflib.SequenceMatcher(None, target_clean, pred_clean).ratio()
        
        if similarity >= threshold:
            similar_words.append((pred, similarity))
    
    return sorted(similar_words, key=lambda x: x[1], reverse=True)

🧪 Testing find_similar_words function:
Target word: 'việt'
Candidates: ['Việt', 'viêt', 'viet', 'vietnam', 'anh', 'việt']
Similar words (threshold=0.5):
  'Việt' -> similarity: 1.000
  'viêt' -> similarity: 1.000
  'viet' -> similarity: 1.000
  'việt' -> similarity: 1.000
  'vietnam' -> similarity: 0.727


In [ ]:
# Test with example
print("🧪 Testing find_similar_words function:")
target = "việt"
candidates = ["Việt", "viêt", "viet", "vietnam", "anh", "việt"]
results = find_similar_words(target, candidates, threshold=0.5)

print(f"Target word: '{target}'")
print(f"Candidates: {candidates}")
print(f"Similar words (threshold=0.5):")
for word, score in results:
    print(f"  '{word}' -> similarity: {score:.3f}")

## 7. Main Function:

In [ ]:
from typing import Dict, Any

# Try with the sentence containing errors
TEST_SENTENCES = "Tôi thích ăn phở vò buổi sáng"

def suggest_spelling_corrections(sentence: str, threshold: float = 1.5) -> Dict[str, Any]:
    """
    Main function to detect spelling errors and suggest corrections.
    
    Args:
        sentence (str): Input sentence to check
        threshold (float): Probability threshold for error detection
        
    Returns:
        Dict[str, Any]: Complete analysis results
    """
    print(f"🔍 Analyzing sentence: '{sentence}'")
    print("-" * 60)
    
    # Step 1: Check for spelling errors
    error_results = check_spelling_errors(sentence, threshold=threshold)
    
    # Filter only the errors (is_error = True)
    errors = [(word, prob, i) for i, (word, prob, is_error) in enumerate(error_results) if is_error]
    
    if not errors:
        print("No spelling errors detected!")
        return {
            'sentence': sentence,
            'errors_found': False,
            'error_details': [],
            'suggestions': {}
        }
    
    print(f"Found {len(errors)} potential spelling error(s):")
    
    suggestions = {}
    error_details = []
    
    # Step 2: Generate suggestions for each error.
    
    for word, prob_score, position in errors:
        print(f"\nWord '{word}' at position {position} (score: {prob_score:.3f}):")
        
        # Get masked sentence
        words = sentence.split()
        masked_sentence = ' '.join(words[:position] + ['[MASK]'] + words[position+1:])
        
        # Get top predictions
        predictions = get_top_predictions(masked_sentence, top_k=10)
        
        # Find similar words
        pred_words = [pred[0] for pred in predictions]  # Extract words from (word, probability) tuples
        similar_words = find_similar_words(word, pred_words, threshold=0.4)
        
        print(f"  🔄 Top suggestions:")
        suggestion_list = []
        for i, (suggestion, similarity) in enumerate(similar_words[:5]):
            pred_prob = next((p[1] for p in predictions if p[0] == suggestion), 0.0)
            print(f"    {i+1}. '{suggestion}' (similarity: {similarity:.3f}, fit: {pred_prob:.3f})")
            suggestion_list.append({
                'word': suggestion,
                'similarity': similarity,
                'probability': pred_prob,
                'confidence': (similarity + pred_prob) / 2
            })
        
        suggestions[word] = suggestion_list
        error_details.append({
            'word': word,
            'position': position,
            'probability_score': prob_score,
            'suggestions': suggestion_list
        })
    
    return {
        'sentence': sentence,
        'errors_found': True,
        'error_details': error_details,
        'suggestions': suggestions
    }
    

# Example usage
test_sentence = TEST_SENTENCES
results = suggest_spelling_corrections(test_sentence, threshold=0.00001)

🔍 Analyzing sentence: 'Tôi thích ăn phở vò buổi sáng'
------------------------------------------------------------
🔍 Analyzing sentence: 'Tôi thích ăn phở vò buổi sáng'
📏 Using threshold: 1e-05
🤖 Model: RobertaForMaskedLM
Word: 'Tôi         ' | Prob: 0.048750 | Confidence: HIGH   | ✅ OK
Word: 'thích       ' | Prob: 0.783203 | Confidence: HIGH   | ✅ OK
Word: 'ăn          ' | Prob: 0.932265 | Confidence: HIGH   | ✅ OK
Word: 'phở         ' | Prob: 0.000099 | Confidence: LOW    | ✅ OK
Word: 'vò          ' | Prob: 0.000006 | Confidence: LOW    | ❌ POTENTIAL ERROR
Word: 'buổi        ' | Prob: 0.647286 | Confidence: HIGH   | ✅ OK
Word: 'sáng        ' | Prob: 0.622689 | Confidence: HIGH   | ✅ OK
📊 Summary: 1/7 words flagged as potential errors (14.3%)
Found 1 potential spelling error(s):
Word: 'buổi        ' | Prob: 0.647286 | Confidence: HIGH   | ✅ OK
Word: 'sáng        ' | Prob: 0.622689 | Confidence: HIGH   | ✅ OK
📊 Summary: 1/7 words flagged as potential errors (14.3%)
Found 1 potential sp

## 8. Evaluation on VSEC test data

Run the detector on a sample of `testdata/VSEC.jsonl` and measure how well the probability threshold finds the annotated spelling errors (syllable level).

In [ ]:
import json, contextlib, io

def load_vsec(path: str, n: int):
    samples = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            ann = item["annotations"]
            words = [a["current_syllable"] for a in ann]
            labels = [not a["is_correct"] for a in ann]   # True = spelling error
            samples.append((words, labels))
            if len(samples) >= n:
                break
    return samples

N_EVAL = int(os.environ.get("VSEC_N_EVAL", 100))
vsec = load_vsec("testdata/VSEC.jsonl", N_EVAL)
print(f"Loaded {len(vsec)} sentences, {sum(sum(l) for _, l in vsec)} annotated errors")

# Collect the probability of every word once, then try several thresholds
all_probs, all_labels = [], []
for words, labels in vsec:
    with contextlib.redirect_stdout(io.StringIO()):   # silence the per-word printing
        res = check_spelling_errors(" ".join(words), threshold=0.0)
    all_probs += [p for _, p, _ in res]
    all_labels += labels

print(f"{'threshold':>10} | {'precision':>9} | {'recall':>6} | {'F1':>6}")
for th in [1e-6, 1e-5, 1e-4, 1e-3, 1e-2]:
    pred = [p < th for p in all_probs]
    tp = sum(a and b for a, b in zip(pred, all_labels))
    fp = sum(a and not b for a, b in zip(pred, all_labels))
    fn = sum(b and not a for a, b in zip(pred, all_labels))
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    print(f"{th:>10g} | {prec:9.3f} | {rec:6.3f} | {f1:6.3f}")
